In [1]:
from numpy import sqrt, asarray
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN
from sklearn.metrics import r2_score
from pandas import read_csv

In [2]:
# load the dataset
filename = '/content/LSTM-Multivariate_pollution.csv'
all_attributes = ['pollution','dew', 'temp', 'press', 'wnd_dir', 'wnd_spd', 'snow', 'rain']
data = read_csv(filename, index_col=False, usecols=all_attributes, encoding='utf-8')[all_attributes]
print(data)

       pollution  dew  temp   press wnd_dir  wnd_spd  snow  rain
0            129  -16  -4.0  1020.0      SE     1.79     0     0
1            148  -15  -4.0  1020.0      SE     2.68     0     0
2            159  -11  -5.0  1021.0      SE     3.57     0     0
3            181   -7  -5.0  1022.0      SE     5.36     1     0
4            138   -7  -5.0  1022.0      SE     6.25     2     0
...          ...  ...   ...     ...     ...      ...   ...   ...
43795          8  -23  -2.0  1034.0      NW   231.97     0     0
43796         10  -22  -3.0  1034.0      NW   237.78     0     0
43797         10  -22  -3.0  1034.0      NW   242.70     0     0
43798          8  -22  -4.0  1034.0      NW   246.72     0     0
43799         12  -21  -3.0  1034.0      NW   249.85     0     0

[43800 rows x 8 columns]


In [3]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
data['wnd_dir'] = encoder.fit_transform(data['wnd_dir'])

data = data.values

print(data)

[[129.   -16.    -4.   ...   1.79   0.     0.  ]
 [148.   -15.    -4.   ...   2.68   0.     0.  ]
 [159.   -11.    -5.   ...   3.57   0.     0.  ]
 ...
 [ 10.   -22.    -3.   ... 242.7    0.     0.  ]
 [  8.   -22.    -4.   ... 246.72   0.     0.  ]
 [ 12.   -21.    -3.   ... 249.85   0.     0.  ]]


In [4]:
print(data[:5])

[[ 1.290e+02 -1.600e+01 -4.000e+00  1.020e+03  2.000e+00  1.790e+00
   0.000e+00  0.000e+00]
 [ 1.480e+02 -1.500e+01 -4.000e+00  1.020e+03  2.000e+00  2.680e+00
   0.000e+00  0.000e+00]
 [ 1.590e+02 -1.100e+01 -5.000e+00  1.021e+03  2.000e+00  3.570e+00
   0.000e+00  0.000e+00]
 [ 1.810e+02 -7.000e+00 -5.000e+00  1.022e+03  2.000e+00  5.360e+00
   1.000e+00  0.000e+00]
 [ 1.380e+02 -7.000e+00 -5.000e+00  1.022e+03  2.000e+00  6.250e+00
   2.000e+00  0.000e+00]]


In [5]:
n = len(data)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

train_data = data[:n_train]
val_data = data[n_train:n_train + n_val]
test_data = data[n_train + n_val:]

print(f"Tổng số mẫu: {n}")
print(f"Tập train (70%): {train_data.shape}")
print(f"Tập val (15%): {val_data.shape}")
print(f"Tập test (15%): {test_data.shape}")

Tổng số mẫu: 43800
Tập train (70%): (30659, 8)
Tập val (15%): (6570, 8)
Tập test (15%): (6571, 8)


In [6]:
# import numpy as np

# def split_sequences(data, n_steps, n_pred, pred_col):
#     X, y = [], []
#     pred_idx = all_attributes.index(pred_col)

#     for i in range(len(data) - n_steps - n_pred + 1):
#         X.append(data[i:(i + n_steps), :])
#         y.append(data[i + n_steps + n_pred - 1, pred_idx])
#     return np.array(X), np.array(y)

# n_steps = 3
# n_pred = 2
# pred_col = "pollution"

# X_train, y_train = split_sequences(train_data, n_steps, n_pred, pred_col)
# X_val, y_val = split_sequences(val_data, n_steps, n_pred, pred_col)
# X_test, y_test = split_sequences(test_data, n_steps, n_pred, pred_col)

In [7]:
# print(X_train[0])
# print(y_train[0])

In [8]:
# print(X_train[-1])
# print(y_train[-1])

In [9]:
# X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2])
# X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], X_val.shape[2])
# X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2])

In [10]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import SimpleRNN, Dense, Input
# from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
# from tensorflow.keras.optimizers import Adam
# from sklearn.metrics import r2_score
# from math import sqrt

# model = Sequential()
# model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
# model.add(SimpleRNN(128, activation='tanh', kernel_initializer='glorot_uniform', return_sequences=True))
# model.add(SimpleRNN(64, activation='tanh', kernel_initializer='glorot_uniform'))
# model.add(Dense(32, activation='relu'))
# model.add(Dense(1))

# model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

# early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
# reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6)

# model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1, validation_data=(X_val, y_val), callbacks=[early_stop, reduce_lr], shuffle=False)

# mse, mae = model.evaluate(X_test, y_test, verbose=0)

# print('MSE: %.3f, RMSE: %.3f, MAE: %.3f' %
#       (mse, sqrt(mse), mae))

# y_pred = model.predict(X_test, verbose=0).reshape(-1)

# print(y_pred)
# print("R2:", r2_score(y_test, y_pred))

In [11]:

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from math import sqrt

# TASK 2

def split_sequences(data, n_steps, n_pred, pred_col):
    X, y = [], []
    pred_idx = all_attributes.index(pred_col)

    for i in range(len(data) - n_steps - n_pred + 1):
        end_ix = i + n_steps

        # Input: previous 3 hours, all 8 features
        X.append(data[i:end_ix, :])

        # Output: pollution of the next 2 hours
        y.append(data[end_ix:end_ix + n_pred, pred_idx])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


n_steps = 3
n_pred = 2
pred_col = "pollution"

X_train, y_train = split_sequences(train_data, n_steps, n_pred, pred_col)
X_val, y_val = split_sequences(val_data, n_steps, n_pred, pred_col)
X_test, y_test = split_sequences(test_data, n_steps, n_pred, pred_col)

In [14]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nX_train[0]:")
print(X_train[0])
print("y_train[0]:")
print(y_train[0])

X_train: (30655, 3, 8)
y_train: (30655, 2)
X_val: (6566, 3, 8)
y_val: (6566, 2)
X_test: (6567, 3, 8)
y_test: (6567, 2)

X_train[0]:
[[ 129.    -16.     -4.   1020.      2.      1.79    0.      0.  ]
 [ 148.    -15.     -4.   1020.      2.      2.68    0.      0.  ]
 [ 159.    -11.     -5.   1021.      2.      3.57    0.      0.  ]]
y_train[0]:
[181. 138.]


In [13]:
model = Sequential()
model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))

model.add(SimpleRNN(
    128,
    activation='tanh',
    kernel_initializer='glorot_uniform',
    return_sequences=True
))

model.add(SimpleRNN(
    64,
    activation='tanh',
    kernel_initializer='glorot_uniform'
))

model.add(Dense(32, activation='relu'))

model.add(Dense(2))

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.summary()

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-6
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    verbose=1,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr]
)

mse, mae = model.evaluate(X_test, y_test, verbose=0)

print('\nOverall:')
print('MSE: %.3f, RMSE: %.3f, MAE: %.3f' %
      (mse, sqrt(mse), mae))

y_pred = model.predict(X_test, verbose=0)

print("Prediction shape:", y_pred.shape)
print("R2 overall:", r2_score(y_test, y_pred))

for hour in range(n_pred):
    y_true_hour = y_test[:, hour]
    y_pred_hour = y_pred[:, hour]

    mse_hour = mean_squared_error(y_true_hour, y_pred_hour)
    mae_hour = mean_absolute_error(y_true_hour, y_pred_hour)
    r2_hour = r2_score(y_true_hour, y_pred_hour)

    print(f"\nHour t+{hour + 1}:")
    print("MSE: %.3f, RMSE: %.3f, MAE: %.3f, R2: %.3f" %
          (mse_hour, sqrt(mse_hour), mae_hour, r2_hour))

print("\nFirst 5 predictions:")
for i in range(5):
    print(f"Sample {i + 1}")
    print("Actual:   ", y_test[i])
    print("Predicted:", y_pred[i])


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 3, 128)         │        17,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,034 (125.13 KB)

 Trainable params: 32,034 (125.13 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - loss: 5307.8115 - mae: 40.6960 - val_loss: 1369.8776 - val_mae: 20.8785 - learning_rate: 0.0010
Epoch 2/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 1372.1331 - mae: 21.0664 - val_loss: 1204.0057 - val_mae: 20.7510 - learning_rate: 0.0010
Epoch 3/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1276.6652 - mae: 20.4588 - val_loss: 1037.0139 - val_mae: 18.5880 - learning_rate: 0.0010
Epoch 4/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1248.7263 - mae: 20.1089 - val_loss: 1126.1913 - val_mae: 20.3934 - learning_rate: 0.0010
Epoch 5/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 1274.8306 - mae: 20.5662 - val_loss: 1091.6625 - val_mae: 19.4091 - learning_rate: 0.0010
Epoch 6/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 1295.3037 - mae: 20.5607 - val_loss: 1096.8362 - val_mae: 19.8303 - learning_rate: 0.0010
Epoch 7/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1223.0044 - mae: 19.8271 - val_l